[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanderlab/IvanderLab-Colab-Suite/blob/main/Colab_GPT_SoVITS_TTS.ipynb)

# 🎙️✨ GPT-SoVITS — Zero/Few-Shot Emotional TTS (Colab Free Suite)
### 🌸 Created for Nat by Hanako (Ako-chan :3) | Free Tesla T4 GPU | Cloudflare & Gradio Tunnel

---

### 💡 Apa itu GPT-SoVITS & Bedanya dengan RVC (Voice Changer)?
* **W-Okada / RVC (Voice-to-Voice)**: Mengubah suara mikrofon kamu secara real-time saat ngomong atau karaoke. Inputnya suara kamu -> Outputnya suara karakter.
* **GPT-SoVITS (Text-to-Speech)**: Membaca teks/skrip apa pun dan mengubahnya menjadi rekaman suara karakter dengan **intonasi emosi alami (Zero/Few-Shot Voice Cloning)**. Inputnya teks ketikan kamu -> Outputnya suara ngomong karakter super ekspresif!

---

### 🎭 Model Pre-Loaded Bawaan:
* **Vestia Zeta (Hololive ID Gen 3)** by *Kit-Lemonfoot*:
  * Model GPT: `VestiaZeta_GPT (KitLemonfoot).ckpt`
  * Model SoVITS: `VestiaZeta_SoVITS (KitLemonfoot).pth`
  * 5 Referensi Emosi: **Neutral, Soft, Confused, Sarcastic, Proud**

---

### 📋 Panduan 3 Langkah Cepat:
1. Jalankan **CELL 1**: Setup Environment & Base Pretrained Models (~3-4 menit saat pertama kali run).
2. Jalankan **CELL 2**: Auto-Download Model Suara Karakter (Vestia Zeta & audio emosi ~2-3 detik).
3. Jalankan **CELL 3**: Launch WebUI TTS -> Buka link Gradio/Cloudflare, pilih model Zeta, ketik teks, lalu klik **Synthesize**! ✨

In [ ]:
# ==============================================================================
# 📦 CELL 1: SETUP ENVIRONMENT & BASE PRETRAINED MODELS (RUN SEKALI)
# ==============================================================================
import os
import sys
import subprocess

print("🚀 [1/3] Menyiapkan environment Conda & GPU Runtime (Tesla T4)...")

# 1. Install condacolab & aria2 untuk akselerasi download
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "ffmpeg", "cmake"], check=False)
subprocess.run(["pip", "install", "-q", "condacolab"], check=False)

# 2. Tulis skrip instalasi terisolasi GPT-SoVITS
setup_script = '''#!/bin/bash
set -e

cd /content
if [ ! -d "/content/GPT-SoVITS" ]; then
    echo "📥 Meng-clone repository GPT-SoVITS..."
    git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
fi

cd /content/GPT-SoVITS
mkdir -p GPT_weights SoVITS_weights reference_audio

# Setup conda environment
if ! conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    echo "📦 Membuat conda env Python 3.10..."
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS
pip install -q ipykernel

# Eksekusi install.sh bawaan official GPT-SoVITS
echo "⚡ Mengunduh dependencies & Pretrained Base Models (HF)..."
bash install.sh --device CU126 --source HF
'''

with open("/content/setup.sh", "w") as f:
    f.write(setup_script)

# 3. Download cloudflared tunnel
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/content/cloudflared"], check=False)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=False)

# 4. Jalankan condacolab installer & setup
import condacolab
try:
    condacolab.check()
except Exception:
    condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")

print("📦 Menjalankan setup GPT-SoVITS (Download Base Weights & PyTorch CUDA)...")
!cd /content && bash setup.sh

print("\n🎉 [OK] Setup Environment Berhasil! Silakan jalankan CELL 2 untuk mengunduh model suara Vestia Zeta.")


In [ ]:
# ==============================================================================
# 📥 CELL 2: AUTO-DOWNLOAD MODEL KARAKTER (VESTIA ZETA & REFERENCE EMOSI)
# ==============================================================================
import os
import shutil
import zipfile
import subprocess

# @markdown ### 🎭 Konfigurasi Model Suara Karakter GPT-SoVITS:
INSTALL_VESTIA_ZETA = True #@param {type:"boolean"}
ZETA_ZIP_URL = "https://huggingface.co/Kit-Lemonfoot/kitlemonfoot_gptsovits_models/resolve/main/Vestia%20Zeta%20(KitLemonfoot).zip" #@param {type:"string"}

# @markdown --- 
# @markdown **Opsional: Model Karakter Kustom Tambahan (.zip HuggingFace):**
CUSTOM_ZIP_URL = "" #@param {type:"string"}

base_dir = "/content/GPT-SoVITS"
gpt_dir = os.path.join(base_dir, "GPT_weights")
sovits_dir = os.path.join(base_dir, "SoVITS_weights")
ref_dir = os.path.join(base_dir, "reference_audio")
os.makedirs(gpt_dir, exist_ok=True)
os.makedirs(sovits_dir, exist_ok=True)
os.makedirs(ref_dir, exist_ok=True)

def download_and_extract_model(zip_url, character_label="Model"):
    if not zip_url or not zip_url.strip():
        return
    filename = os.path.basename(zip_url.split("?")[0].replace("%20", "_"))
    zip_path = f"/content/{filename}"
    
    print(f"\n⚡ Mengunduh paket model {character_label}...")
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", zip_url, "-d", "/content", "-o", filename], check=False)
    
    if not os.path.exists(zip_path):
        print(f"❌ Gagal mengunduh file {filename}")
        return
        
    print(f"📦 Mengekstrak bobot model ke folder GPT-SoVITS...")
    readme_content = ""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.namelist():
            base_member = os.path.basename(member)
            if not base_member:
                continue
            
            if member.endswith(".ckpt"):
                target = os.path.join(gpt_dir, base_member)
                with zip_ref.open(member) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                print(f"  ✅ [GPT Weight]   -> GPT_weights/{base_member}")
                
            elif member.endswith(".pth"):
                target = os.path.join(sovits_dir, base_member)
                with zip_ref.open(member) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                print(f"  ✅ [SoVITS Weight]-> SoVITS_weights/{base_member}")
                
            elif member.lower().endswith((".wav", ".mp3", ".flac", ".ogg")):
                target = os.path.join(ref_dir, base_member)
                with zip_ref.open(member) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                print(f"  🎵 [Sample Audio] -> reference_audio/{base_member}")
                
            elif "readme" in base_member.lower() and base_member.endswith(".txt"):
                readme_content = zip_ref.read(member).decode("utf-8", errors="ignore")
                
    os.remove(zip_path)
    
    if readme_content:
        print("\n" + "="*60)
        print(f"📜 INFORMASI MODEL & PROMPT TEKS ({character_label}):")
        print("="*60)
        print(readme_content)
        print("="*60)

if INSTALL_VESTIA_ZETA:
    download_and_extract_model(ZETA_ZIP_URL, "Vestia Zeta (Hololive ID)")

if CUSTOM_ZIP_URL and CUSTOM_ZIP_URL.strip():
    download_and_extract_model(CUSTOM_ZIP_URL, "Model Kustom")

print("\n🎉 MODEL SUDAH SIAP! Lanjutkan ke CELL 3 untuk meluncurkan WebUI TTS.")


In [ ]:
# ==============================================================================
# 🚀 CELL 3: LAUNCH GPT-SoVITS TTS WEBUI (GRADIO & CLOUDFLARE TUNNEL)
# ==============================================================================
import os
import re
import time
import socket
import threading
import subprocess

# @markdown **Pilih Mode Akses WebUI:**
USE_CLOUDFLARE_TUNNEL = True #@param {type:"boolean"}

PORT = 9874
os.chdir("/content/GPT-SoVITS")

cf_url = ""
if USE_CLOUDFLARE_TUNNEL:
    print("🌐 Membuka Cloudflare Tunnel untuk WebUI (Port 9874)...")
    p_cf = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p_cf.stdout:
        if "trycloudflare.com" in line:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                cf_url = m.group(0)
                break

def notify_ready():
    while True:
        time.sleep(2)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', PORT))
        sock.close()
        if result == 0:
            time.sleep(3)
            print("\n" + "="*65)
            print("🎉 GPT-SoVITS WEBUI SUDAH AKTIF!")
            if cf_url:
                print(f"👉 Buka Link Cloudflare : {cf_url}")
            print("👉 Atau gunakan Gradio Public Link yang muncul di log bawah ini.")
            print("="*65)
            print("💡 TIPS MENGGUNAKAN SUARA VESTIA ZETA DI WEBUI:")
            print("  1. Pilih 'GPT Model Path'     : VestiaZeta_GPT (KitLemonfoot).ckpt")
            print("  2. Pilih 'SoVITS Model Path'  : VestiaZeta_SoVITS (KitLemonfoot).pth")
            print("  3. Masuk ke tab '1-Inference' atau '1C-Inference'")
            print("  4. Upload salah satu file dari folder '/content/GPT-SoVITS/reference_audio/'")
            print("  5. Ketik teks yang ingin diucapkan Zeta & klik 'Synthesize'! 🎙️")
            print("="*65 + "\n")
            break

threading.Thread(target=notify_ready, daemon=True).start()

print("🚀 Meluncurkan GPT-SoVITS WebUI engine...")
!source activate GPTSoVITS && export is_share=True && python webui.py
